In [ ]:
import re
import collections
import urllib.request
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

keras.utils.set_random_seed(42)
np.random.seed(42)

In [ ]:
text_url = "https://storage.googleapis.com/download.tensorflow.org/data/shakespeare.txt"
urllib.request.urlretrieve(text_url, "shakespeare.txt")

with open("shakespeare.txt", "r", encoding="utf-8") as text_file:
    raw_text = text_file.read()

print("characters:", len(raw_text))
print(raw_text[:500])

In [ ]:
def tokenize(line):
    return re.findall(r"[a-z]+(?:'[a-z]+)?", line.lower())

sentences = [tokenize(line) for line in raw_text.splitlines()]
sentences = [sentence for sentence in sentences if len(sentence) >= 2]
word_counts = collections.Counter(word for sentence in sentences for word in sentence)
total_words = sum(word_counts.values())

print("sentences:", len(sentences))
print("total words:", total_words)
print("unique words:", len(word_counts))
print(word_counts.most_common(20))

In [ ]:
vocab_size = 4000
vocabulary = ["<unk>"] + [word for word, _ in word_counts.most_common(vocab_size - 1)]
word_to_index = {word: index for index, word in enumerate(vocabulary)}
encoded_sentences = [[word_to_index.get(word, 0) for word in sentence] for sentence in sentences]
word_frequencies = np.array([word_counts.get(word, 0) for word in vocabulary], dtype=np.float64) / total_words

print(sentences[0])
print(encoded_sentences[0])

In [ ]:
subsample_threshold = 1e-3
keep_probability = np.minimum(1.0, (np.sqrt(word_frequencies / subsample_threshold) + 1) * subsample_threshold / np.maximum(word_frequencies, 1e-12))
keep_probability[0] = 0.0

def subsample_sentence(sentence):
    return [word for word in sentence if np.random.rand() < keep_probability[word]]

for word in ["the", "and", "i", "love", "king"]:
    print(f"{word:<6} keep probability {keep_probability[word_to_index[word]]:.3f}")

In [ ]:
window_size = 3

def build_skipgram_pairs(encoded_sentences):
    target_words, context_words = [], []
    for sentence in encoded_sentences:
        sentence = subsample_sentence(sentence)
        for position, target_word in enumerate(sentence):
            current_window = np.random.randint(1, window_size + 1)
            start = max(0, position - current_window)
            end = min(len(sentence), position + current_window + 1)
            for context_position in range(start, end):
                if context_position != position:
                    target_words.append(target_word)
                    context_words.append(sentence[context_position])
    return np.array(target_words, dtype=np.int32), np.array(context_words, dtype=np.int32)

target_words, context_words = build_skipgram_pairs(encoded_sentences)
print("training pairs:", len(target_words))
for target_word, context_word in list(zip(target_words, context_words))[:12]:
    print(f"({vocabulary[target_word]}, {vocabulary[context_word]})")

In [ ]:
negative_samples = 5
noise_distribution = word_frequencies ** 0.75
noise_distribution[0] = 0
noise_distribution = noise_distribution / noise_distribution.sum()

def add_negative_samples(target_words, context_words):
    negative_words = np.random.choice(vocab_size, size=(len(target_words), negative_samples), p=noise_distribution).astype(np.int32)
    candidate_words = np.concatenate([context_words[:, None], negative_words], axis=1)
    candidate_labels = np.zeros(candidate_words.shape, dtype=np.float32)
    candidate_labels[:, 0] = 1.0
    return candidate_words, candidate_labels

candidate_words, candidate_labels = add_negative_samples(target_words, context_words)
example = 0
print("target word :", vocabulary[target_words[example]])
print("candidates  :", [vocabulary[word] for word in candidate_words[example]])
print("labels      :", candidate_labels[example])

In [ ]:
embedding_dim = 100

target_input = keras.Input(shape=(1,), dtype="int32", name="target_word")
candidate_input = keras.Input(shape=(negative_samples + 1,), dtype="int32", name="candidate_words")
target_vector = layers.Embedding(vocab_size, embedding_dim, name="word_embedding")(target_input)
candidate_vectors = layers.Embedding(vocab_size, embedding_dim, name="context_embedding")(candidate_input)
similarity_scores = layers.Dot(axes=(2, 2))([candidate_vectors, target_vector])
logits = layers.Flatten()(similarity_scores)

word2vec_model = keras.Model(inputs=[target_input, candidate_input], outputs=logits, name="skipgram_negative_sampling")
word2vec_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss=keras.losses.CategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"]
)
word2vec_model.summary()

In [ ]:
epochs = 15
batch_size = 1024
loss_history, accuracy_history = [], []

for epoch in range(epochs):
    target_words, context_words = build_skipgram_pairs(encoded_sentences)
    candidate_words, candidate_labels = add_negative_samples(target_words, context_words)
    epoch_result = word2vec_model.fit(
        [target_words[:, None], candidate_words], candidate_labels,
        batch_size=batch_size, epochs=1, shuffle=True, verbose=0
    )
    loss_history.append(epoch_result.history["loss"][0])
    accuracy_history.append(epoch_result.history["accuracy"][0])
    print(f"epoch {epoch + 1:>2}/{epochs}  pairs {len(target_words)}  loss {loss_history[-1]:.4f}  accuracy {accuracy_history[-1]:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(loss_history, marker="o")
axes[0].set_title("Negative sampling loss")
axes[0].set_xlabel("epoch")
axes[1].plot(accuracy_history, marker="o", color="green")
axes[1].set_title("True context picked over negatives")
axes[1].set_xlabel("epoch")
plt.show()

In [ ]:
word_vectors = word2vec_model.get_layer("word_embedding").get_weights()[0]
normalized_vectors = word_vectors / np.clip(np.linalg.norm(word_vectors, axis=1, keepdims=True), 1e-8, None)

def most_similar(word, top_n=8):
    if word not in word_to_index:
        return []
    scores = normalized_vectors @ normalized_vectors[word_to_index[word]]
    best_matches = np.argsort(-scores)[1:top_n + 1]
    return [(vocabulary[index], round(float(scores[index]), 3)) for index in best_matches]

for query_word in ["king", "love", "death", "night", "good", "father", "sword", "eyes"]:
    print(f"{query_word:<7} -> {most_similar(query_word)}")

In [ ]:
def analogy(word_a, word_b, word_c, top_n=5):
    if not all(word in word_to_index for word in [word_a, word_b, word_c]):
        return []
    query = normalized_vectors[word_to_index[word_b]] - normalized_vectors[word_to_index[word_a]] + normalized_vectors[word_to_index[word_c]]
    scores = normalized_vectors @ (query / np.linalg.norm(query))
    excluded = {word_to_index[word_a], word_to_index[word_b], word_to_index[word_c]}
    ranked = [index for index in np.argsort(-scores) if index not in excluded]
    return [vocabulary[index] for index in ranked[:top_n]]

print("king - man + woman ->", analogy("man", "king", "woman"))
print("father - son + daughter ->", analogy("son", "father", "daughter"))
print("day - light + dark ->", analogy("light", "day", "dark"))

In [ ]:
plot_word_ids = np.arange(1, 151)
pca_points = PCA(n_components=2, random_state=42).fit_transform(word_vectors[plot_word_ids])

plt.figure(figsize=(14, 10))
plt.scatter(pca_points[:, 0], pca_points[:, 1], s=10, color="gray")
for (x_value, y_value), word_id in zip(pca_points, plot_word_ids):
    plt.annotate(vocabulary[word_id], (x_value, y_value), fontsize=8)
plt.title("PCA projection of the 150 most frequent words")
plt.show()

In [ ]:
tsne_word_ids = np.arange(1, 301)
tsne_points = TSNE(n_components=2, perplexity=30, init="pca", random_state=42).fit_transform(word_vectors[tsne_word_ids])

plt.figure(figsize=(16, 12))
plt.scatter(tsne_points[:, 0], tsne_points[:, 1], s=8, color="gray")
for (x_value, y_value), word_id in zip(tsne_points, tsne_word_ids):
    plt.annotate(vocabulary[word_id], (x_value, y_value), fontsize=7)
plt.title("t-SNE projection of the 300 most frequent words")
plt.show()

In [ ]:
word_groups = {
    "royalty": ["king", "queen", "prince", "duke", "lord", "lady", "crown", "throne"],
    "family": ["father", "mother", "son", "daughter", "brother", "sister", "wife", "husband"],
    "body": ["eyes", "hand", "heart", "head", "face", "blood", "tongue", "lips"],
    "time": ["day", "night", "morning", "hour", "time", "year", "today", "tomorrow"],
    "emotion": ["love", "hate", "fear", "joy", "grief", "sorrow", "anger", "hope"]
}
group_words = [(group, word) for group, words in word_groups.items() for word in words if word in word_to_index]
group_vectors = np.array([word_vectors[word_to_index[word]] for _, word in group_words])
group_colors = dict(zip(word_groups.keys(), ["tab:red", "tab:blue", "tab:green", "tab:orange", "tab:purple"]))

group_pca = PCA(n_components=2, random_state=42).fit_transform(group_vectors)
group_tsne = TSNE(n_components=2, perplexity=8, init="pca", random_state=42).fit_transform(group_vectors)

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
for axis, points, title in [(axes[0], group_pca, "PCA"), (axes[1], group_tsne, "t-SNE")]:
    for (group, word), (x_value, y_value) in zip(group_words, points):
        axis.scatter(x_value, y_value, color=group_colors[group], s=40)
        axis.annotate(word, (x_value, y_value), fontsize=10)
    for group, color in group_colors.items():
        axis.scatter([], [], color=color, label=group)
    axis.legend()
    axis.set_title(f"{title} of themed word groups")
plt.show()

In [ ]:
with open("vectors.tsv", "w", encoding="utf-8") as vector_file, open("metadata.tsv", "w", encoding="utf-8") as metadata_file:
    for index, word in enumerate(vocabulary):
        if index == 0:
            continue
        vector_file.write("\t".join(f"{value:.5f}" for value in word_vectors[index]) + "\n")
        metadata_file.write(word + "\n")
print("saved vectors.tsv and metadata.tsv for projector.tensorflow.org")